# 06 · RLM from scratch

**Idea:** never paste the context into the prompt. Put it in a Python variable and let the model *write code* to read it.

| | Direct LLM | RAG | RLM |
|---|---|---|---|
| What the model sees | all records | 20 retrieved records | the question + "`context` has N lines" |
| How it reads the data | at once | at once | step by step, through code it writes |
| Covers every record? | yes, but context rot | no | yes, via loops and sub-calls |

The code is in `src/rlm_logistics/rlm.py`; this notebook shows how it works and runs it on the dev model.

In [1]:
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from rlm_logistics import methods
from rlm_logistics.budget import Budget
from rlm_logistics.llm import LLM, Reply
from rlm_logistics.rlm import RLM

ROOT = Path("..")
load_dotenv(ROOT / ".env")
budget = Budget()   # hard spending cap on all paid calls; the limit is BUDGET_USD in .env
DEV = ROOT / "results/dev"
meta = json.loads((ROOT / "data/processed/contexts.json").read_text())
qs = pd.read_json(ROOT / "data/processed/questions.jsonl", lines=True)
lines = (ROOT / "data/processed/context_2m.txt").read_text(encoding="utf-8").split("\n")
context = "\n".join(lines[:meta["32k"]["n_records"]])
sample = qs[qs["size"] == "32k"].groupby("tier").head(1).to_dict("records")

## 1. The loop, with a scripted model
Before using a real model, a fake one with fixed replies shows the mechanics in under a second:
1. the root model replies with a ```` ```repl ```` code block;
2. we run it and send back what it printed;
3. variables survive between turns; `FINAL_VAR(n)` returns one as the answer.

In [2]:
class Scripted:
    num_ctx, model = 8192, "scripted"
    def __init__(self, *replies): self.replies = list(replies)
    def chat(self, prompt, system="", **kw): return Reply(self.replies.pop(0), 0, 0, 0.0, 0.0)

turn1 = "```repl\nlate = [l for l in context.split(chr(10)) if 'days late' in l]\nprint(len(late), late[0][:120])\n```"
reply = RLM(Scripted(turn1, "FINAL_VAR(late)"), depth=0).answer("Which orders were late?", context)
for step in reply.trace:
    print(f"--- step {step['step']} ---\nmodel wrote:\n{step['response']}\nREPL printed: {step['outputs']}\n")

--- step 0 ---
model wrote:
```repl
late = [l for l in context.split(chr(10)) if 'days late' in l]
print(len(late), late[0][:120])
```
REPL printed: ['29 [order O29877] seller=S0911 (SP/jacarei) customer=ES/vila velha category=electronics purchased=2018-01-16 estimated=2018\n']

--- step 1 ---
model wrote:
FINAL_VAR(late)
REPL printed: []



## 2. What the real root model is told
The system prompt describes the variable, the record format and the label rules, and what the REPL allows.
The context itself (~32K tokens here, 2M later) is **not** in it, so the root prompt stays small at any size.

In [3]:
from rlm_logistics.rlm import SYSTEM, TOOLS, LABEL_RULES, ALLOWED_IMPORTS
print(SYSTEM.format(chars=len(context), lines=len(context.split(chr(10))), rules="(label rules)", out_chars=4000,
                    imports=", ".join(sorted(ALLOWED_IMPORTS)), tools=TOOLS[1].format(sub_chars=29_000)))

You answer a question about a large text you cannot see directly. It is stored in a Python REPL
as the variable `context` (67,948 characters, 269 lines). Each line is one order record:
[order ID] seller=ID (state/city) customer=state/city category=... purchased/estimated/delivered dates (delivery status) review=score/5 "comment in Portuguese"
Questions about complaints use these review labels:
(label rules)

Write Python inside ```repl ... ``` blocks. After each reply you see what the code printed (cut to 4000 characters),
so print what you need. Variables persist between blocks. Allowed imports: collections, functools, itertools, json, math, re, statistics. No files or network.
You can call llm_query(prompt) -> str. It sends `prompt` to a language model that reads at most 29,000
characters at once. Use it for judgments code cannot make, such as reading review comments. Put many records in one
call (e.g. 40 lines with the instruction to answer per line) and loop over chunks, instead of

## 3. Depth 0, 1, 2
* **Depth 0**: code only. Fine for lookups (`find the line with O39999`), but code cannot read Portuguese reviews.
* **Depth 1**: adds `llm_query(prompt)`, a sub-model for judgments code can't make, e.g. "label these 40 reviews".
* **Depth 2**: each `llm_query` is itself an RLM, which can split its own input further.

Safety (PRD): the REPL has no `open`, no network, imports limited to a short list, a time limit per code block
(checked on every instruction), a cap on sub-calls, and a per-question budget.

## 4. Behaviour check on Claude Haiku (32K)
The PRD warns qwen3:8b may be too weak as an RLM root, and on CPU it needs hours, so this check runs on Claude Haiku 5.5
(root **and** sub-calls), under the spending cap: one question per tier at depth 0 and 1, plus depth 2 on T2.
`methods.run` saves every run, so re-running this notebook reuses them and costs nothing.

In [4]:
llm = LLM.from_config("final", budget=budget)
plan = [(f"rlm_depth{d}", d, q) for d in (0, 1) for q in sample] + [("rlm_depth2", 2, sample[1])]
rows = [methods.run(name, q, lambda: methods.rlm(q["question"], llm, context, d), llm.model, DEV)
        for name, d, q in plan]
pd.DataFrame(rows)[["method", "tier", "score", "steps", "sub_calls", "input_tokens", "cost", "seconds"]]

,method,tier,score,steps,sub_calls,input_tokens,cost,seconds
0,rlm_depth0,T1,1.000000,3,0,7209,0.002902,20.4
1,rlm_depth0,T2,1.000000,4,0,6451,0.001708,13.1
2,rlm_depth0,T3,0.765957,6,0,31765,0.007535,42.4
3,rlm_depth1,T1,1.000000,3,0,2870,0.000441,4.8
4,rlm_depth1,T2,1.000000,4,0,7799,0.001829,12.8
5,rlm_depth1,T3,0.727273,4,6,16960,0.006135,60.4
6,rlm_depth2,T2,0.000000,5,0,19620,0.005115,28.2


## 5. Reading a trajectory
Every run saves its full trajectory (`results/dev/trajectories/<method>/<id>.json`): what the model wrote at each step,
what the REPL printed, and every sub-call. This is how RLM failures get diagnosed (RQ5).

In [5]:
trace = json.loads((DEV / "trajectories/rlm_depth1" / f"{sample[1]['id']}.json").read_text(encoding="utf-8"))
for step in trace[:4]:
    print(f"=== step {step['step']}: {len(step['sub_calls'])} sub-calls ===")
    print(step["response"][:700])
    print("REPL ->", (step["outputs"] or ["(none)"])[0][:300], "\n")

=== step 0: 0 sub-calls ===

<repl
print(context[:1500])
import re
lines=context.split('\n')
print(len(lines))
print(lines[0])
print(lines[1])
</invoke>

Ellipsis

Output was truncated, weird. Let me print again.


<repl
print(type(context), len(context))
lines = context.splitlines()
print(len(lines))
for l in lines[:3]: print(repr(l[:400]))
</invoke>

Ellipsis

The output seems to parse oddly. Let me print again carefully.


<repl
print(repr(context)[:300])
</invoke>


The previous code blocks seem to have not actually run; I should write code in a repl block properly.


<repl
print(type(context), len(context))
</invoke>

I need to run the code in a ```repl block; my previous attempts used the wrong format and nothing
REPL -> <class 'str'> 67948
269
'[order O58113] seller=S1642 (SC/blumenau) customer=SP/valinhos category=computers_accessories purchased=2018-02-21 estimated=2018-03-16 delivered=2018-03-09 (7 days early) review=3/5 "Satisfeito"'
'[order O29877] seller=S0911 (SP/jacarei)

## What to notice
* The root prompt stays a few thousand tokens no matter how big `context` is: the size problem moved into code.
* Depth 0 can only do what string matching can, so T2/T3 (which need review labels) mostly need depth 1.
* Cost per run is cents: the root prompt is small and sub-calls only read slices.
* This is a behaviour check on 7 runs, not a result: the full grid is Phase 5.